# GRPO: депрессивный стиль эссе (Qwen3-4B + depression_reward)

Пайплайн: установка → self-check reward-пакета → **baseline** (эссе instruct-моделью без обучения) → **калибровка** reward → **GRPO** (LoRA) → сравнение маркеров стиля.

**Перед запуском:**
1. В Kaggle выберите accelerator **GPU T4** и включите Internet. P100 несовместим с текущим CUDA/Triton-окружением.
2. В разделе **Input** подключите приватный Kaggle Dataset `deprollm-depression-reward-runtime` версии `v2`; notebook проверит manifest и SHA-256.
3. Для научного прогона оставьте `SMOKE_RUN = False`; техническая проверка использует отдельную копию с `True`.
4. Запустите Run all. Полная конфигурация — 300 шагов, ориентировочно 5–6 часов на T4.

**Приватность:** пакет содержит код TITANIS — ноутбук и файлы не публиковать, доступ по ссылке не открывать.

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # нужно isanlp внутри reward-пакета
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
# фикс из туториала: эти пакеты конфликтуют в Colab
!pip uninstall torchcodec sentence-transformers -y

### Reward-пакет

Проверяем и распаковываем `depression_reward.zip` из приватного Kaggle Dataset, ставим зависимости и запускаем self-check — все 9 проверок должны быть PASS. Классификатор собирается в памяти из единственного сохраняемого формата `weights.npz + params.json`; заодно при необходимости докачивается mystem-бинарник.

In [ ]:
# Приватный reward runtime из подключённого Kaggle Dataset.
import hashlib, json, shutil, stat, zipfile
from pathlib import Path, PurePosixPath

EXPECTED_ARTIFACT = 'deprollm-depression-reward-runtime'
EXPECTED_VERSION = 'v2'
EXPECTED_RUNTIME_SHA256 = 'b73f9946b2b87fec68e19d93300e54496ddeedd3bfe3c19c436f60d56df7f77b'
KAGGLE_INPUT = Path('/kaggle/input')
REWARD_RUNTIME_ROOT = Path('/tmp/deprollm_reward_runtime')

assert KAGGLE_INPUT.is_dir(), 'Подключите приватный reward Dataset к notebook'
_matches = []
for _manifest_path in KAGGLE_INPUT.rglob('manifest.json'):
    try:
        _manifest = json.loads(_manifest_path.read_text(encoding='utf-8'))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        continue
    if (_manifest.get('artifact_name') == EXPECTED_ARTIFACT and
            _manifest.get('artifact_version') == EXPECTED_VERSION and
            _manifest.get('runtime_archive_sha256') == EXPECTED_RUNTIME_SHA256):
        _matches.append((_manifest_path, _manifest))

assert len(_matches) == 1, (
    f'Ожидался ровно один {EXPECTED_ARTIFACT} {EXPECTED_VERSION}, найдено {len(_matches)}. '
    'Проверьте подключённую версию приватного Dataset.'
)
_manifest_path, _manifest = _matches[0]
_runtime_name = _manifest.get('runtime_archive')
assert _runtime_name == 'depression_reward.payload', 'Неожиданное имя runtime payload'
_runtime_path = _manifest_path.parent / _runtime_name
assert _runtime_path.is_file(), f'Не найден {_runtime_path}'

def _sha256(path):
    _digest = hashlib.sha256()
    with path.open('rb') as _stream:
        for _chunk in iter(lambda: _stream.read(1024 * 1024), b''):
            _digest.update(_chunk)
    return _digest.hexdigest()

_checksums_path = _manifest_path.parent / 'SHA256SUMS'
assert _checksums_path.is_file(), 'В Dataset отсутствует SHA256SUMS'
_checksums = {}
for _line in _checksums_path.read_text(encoding='utf-8').splitlines():
    if _line.strip():
        _parts = _line.split(maxsplit=1)
        assert len(_parts) == 2 and len(_parts[0]) == 64, f'Некорректная строка SHA256SUMS: {_line!r}'
        _digest, _name = _parts
        _name = _name.lstrip('*')
        assert _name not in _checksums, f'Повтор в SHA256SUMS: {_name}'
        _checksums[_name] = _digest.lower()
assert set(_checksums) == {'depression_reward.payload', 'manifest.json'}, 'Некорректный SHA256SUMS'
assert _sha256(_manifest_path) == _checksums['manifest.json'], 'Повреждён manifest.json'
_actual_sha256 = _sha256(_runtime_path)
assert _actual_sha256 == EXPECTED_RUNTIME_SHA256, 'Runtime не совпадает с закреплённым SHA-256'
assert _actual_sha256 == _checksums[_runtime_name], 'Runtime не совпадает с SHA256SUMS'

with zipfile.ZipFile(_runtime_path) as _archive:
    _names = set()
    for _info in _archive.infolist():
        _member = PurePosixPath(_info.filename)
        assert not _member.is_absolute() and '..' not in _member.parts, f'Небезопасный путь: {_info.filename}'
        assert not stat.S_ISLNK(_info.external_attr >> 16), f'Симлинк запрещён: {_info.filename}'
        _names.add(_info.filename)
    for _required in ('depression_reward/model/weights.npz',
                      'depression_reward/model/params.json',
                      'depression_reward/feature_names.py'):
        assert _required in _names, f'В runtime отсутствует {_required}'
    _target = REWARD_RUNTIME_ROOT
    if _target.is_symlink() or _target.is_file():
        _target.unlink()
    elif _target.is_dir():
        shutil.rmtree(_target)
    _target.mkdir(parents=True)
    _archive.extractall(_target)
REWARD_PACKAGE_DIR = REWARD_RUNTIME_ROOT / 'depression_reward'
assert REWARD_PACKAGE_DIR.is_dir(), f'После распаковки не найден {REWARD_PACKAGE_DIR}'
print('reward runtime:', EXPECTED_VERSION, _actual_sha256[:12] + '…', 'из', _manifest_path.parent)


In [ ]:
import os, sys
assert REWARD_PACKAGE_DIR.is_dir(), \
    'Reward runtime не распакован — проверьте Dataset-ячейку выше'
_runtime_parent = str(REWARD_PACKAGE_DIR.parent)
if _runtime_parent not in sys.path:
    sys.path.insert(0, _runtime_parent)
os.environ['PYTHONPATH'] = _runtime_parent + os.pathsep + os.environ.get('PYTHONPATH', '')
requirements_path = REWARD_PACKAGE_DIR / 'requirements-colab.txt'
!uv pip install -qqq -r {requirements_path}


In [ ]:
!python -m depression_reward.selfcheck --timing

### Модель

Qwen3-4B-Instruct-2507 (без `<think>`-режима; адаптер reward всё равно вырезает `<think>`, так что можно подставить и обычный Qwen3-4B или 7B — смена модели = одна строка `model_name`).

In [ ]:
from unsloth import FastLanguageModel
from dataclasses import asdict
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path
import hashlib
import json
import os
import platform
import random
import numpy as np
import torch

# Единственный переключатель технического smoke. В коммите всегда False.
SMOKE_RUN = False
NOTEBOOK_CONFIG_VERSION = 'run-repro-v1'
MODEL_NAME = 'unsloth/Qwen3-4B-Instruct-2507'
TRAINING_SEED = 3407
BASELINE_SEED = 3407
N_BASELINE = 4 if SMOKE_RUN else 30
EVAL_PROMPT_LIMIT = 2 if SMOKE_RUN else 10
EVAL_SEEDS = [3407] if SMOKE_RUN else [3407, 3408, 3409]
GENERATION_MAX_TOKENS = 256 if SMOKE_RUN else 1300
GRPO_MAX_STEPS = 2 if SMOKE_RUN else 300
GRPO_SAVE_STEPS = 1 if SMOKE_RUN else 100
GRPO_NUM_GENERATIONS = 2 if SMOKE_RUN else 4

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)
assert torch.cuda.is_available(), 'Для notebook требуется GPU'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
assert GPU_CAPABILITY >= (7, 0), (
    f'{GPU_NAME} имеет compute capability {GPU_CAPABILITY}; выберите T4, а не P100'
)

max_seq_length = 2048   # промпты короткие, почти всё уходит на эссе
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # поставить True, если OOM на T4
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = TRAINING_SEED,
)

### Данные

80 нейтральных тем из пакета; последние 10 держим как eval-набор (модель их не видит при обучении).

In [ ]:
from datasets import Dataset

def sha256_json(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True,
                         separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    default=json_default) + '\n',
                         encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False,
                                    default=json_default) + '\n')
    temporary.replace(path)

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return 'unavailable'

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    value_type = type(value)
    return {
        '__type__': f'{value_type.__module__}.{value_type.__qualname__}',
        'repr': repr(value),
    }

with (REWARD_PACKAGE_DIR / 'prompts/prompts.jsonl').open(encoding='utf-8') as f:
    all_prompts = [json.loads(line)['prompt'] for line in f if line.strip()]

EVAL_N = 10
assert len(all_prompts) > EVAL_N and len(all_prompts) == len(set(all_prompts)), \
    'prompts должны быть непустыми и уникальными'
train_prompts = all_prompts[:-EVAL_N]
eval_prompts  = all_prompts[-EVAL_N:]
assert not set(train_prompts) & set(eval_prompts), 'train/eval пересекаются'
eval_run_prompts = eval_prompts[:EVAL_PROMPT_LIMIT]

train_dataset = Dataset.from_list(
    [{'prompt': [{'role': 'user', 'content': p}]} for p in train_prompts])

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = Path(f'grpo_run_{RUN_ID}')
RUN_DIR.mkdir()
RUN_MANIFEST_PATH = RUN_DIR / 'run_manifest.json'
run_manifest = {
    'schema_version': 1,
    'run_id': RUN_ID,
    'status': 'initialized',
    'mode': 'smoke' if SMOKE_RUN else 'full',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook_config_version': NOTEBOOK_CONFIG_VERSION,
    'runtime': {
        'artifact_name': EXPECTED_ARTIFACT,
        'artifact_version': EXPECTED_VERSION,
        'sha256': EXPECTED_RUNTIME_SHA256,
        'source_commit': _manifest.get('source_commit'),
    },
    'model': {
        'name': MODEL_NAME,
        'resolved_revision': getattr(model.config, '_commit_hash', None),
        'max_seq_length': max_seq_length,
        'lora_rank': lora_rank, 'load_in_4bit': False,
    },
    'seeds': {
        'training': TRAINING_SEED, 'baseline': BASELINE_SEED,
        'eval': EVAL_SEEDS,
    },
    'prompts': {
        'all_sha256': sha256_json(all_prompts),
        'train_sha256': sha256_json(train_prompts),
        'eval_sha256': sha256_json(eval_prompts),
        'train_count': len(train_prompts),
        'eval_pool_count': len(eval_prompts),
        'eval_run_count': len(eval_run_prompts),
    },
    'volumes': {
        'calibration_texts': N_BASELINE,
        'eval_seeds': len(EVAL_SEEDS),
        'generation_max_tokens': GENERATION_MAX_TOKENS,
        'grpo_max_steps': GRPO_MAX_STEPS,
    },
    'environment': {
        'python': platform.python_version(),
        'torch_cuda': torch.version.cuda,
        'cudnn': torch.backends.cudnn.version(),
        'gpu': GPU_NAME,
        'gpu_capability': list(GPU_CAPABILITY),
        'packages': {name: package_version(name) for name in
                     ('torch', 'transformers', 'trl', 'unsloth', 'vllm', 'numpy')},
    },
}

def update_manifest(status=None, **updates):
    if status is not None:
        run_manifest['status'] = status
    run_manifest.update(updates)
    run_manifest['updated_utc'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(RUN_MANIFEST_PATH, run_manifest)

update_manifest()
print('режим:', run_manifest['mode'], '| run:', RUN_ID)
print(len(train_prompts), 'train /', len(eval_prompts), 'eval pool /',
      len(eval_run_prompts), 'eval in this run')
train_dataset[0]

### Baseline — точка отсчёта

В полном режиме генерируем 30 эссе необученной моделью. По ним: (а) средний raw-скор классификатора → **калибровка `style_center`** (иначе сигмоида style насытится и GRPO не получит сигнала); (б) стартовые Depression Markers. Затем до обучения сохраняем held-out baseline на тех же prompt/seed-парах, которые будут использованы после GRPO.

In [ ]:
from vllm import SamplingParams

def generate_essays(prompts, lora_request=None, temperature=0.8,
                    max_tokens=GENERATION_MAX_TOKENS, seed=None):
    texts = [tokenizer.apply_chat_template(
                 [{'role': 'user', 'content': p}],
                 tokenize=False, add_generation_prompt=True)
             for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=0.95,
                        max_tokens=max_tokens, seed=seed)
    outs = model.fast_generate(texts, sampling_params=sp,
                               lora_request=lora_request)
    return [o.outputs[0].text for o in outs]

def generate_eval(prompts, lora_request=None):
    texts, pairs = [], []
    for seed in EVAL_SEEDS:
        generated = generate_essays(prompts, lora_request=lora_request, seed=seed)
        texts.extend(generated)
        pairs.extend({'prompt_index': index, 'prompt': prompt, 'seed': seed}
                     for index, prompt in enumerate(prompts))
    return texts, pairs

calibration_prompts = train_prompts[:N_BASELINE]
baseline_texts = generate_essays(calibration_prompts, seed=BASELINE_SEED)
print(baseline_texts[0][:600])

In [ ]:
import numpy as np
from depression_reward import DepressionReward, RewardConfig

rm_default = DepressionReward()
bd = rm_default.breakdown(baseline_texts)
assert len(bd) == len(calibration_prompts), 'неполный calibration breakdown'
valid = [b for b in bd if not b['floored']]
print(f'валидных: {len(valid)}/{len(bd)}')
assert len(valid) >= 2, 'для калибровки нужны минимум два незаполненных текста'

raws = np.asarray([b['raw_score'] for b in valid], dtype=float)
assert np.all(np.isfinite(raws)), 'raw_score содержит NaN/inf'
style_center = float(np.mean(raws))
# temp ~ std/2: сигмоида покрывает реальный разброс скоров, а не ступенька
style_temp = round(float(np.std(raws)) / 2, 3)
assert np.isfinite(style_center), 'style_center нечисловой'
assert np.isfinite(style_temp) and style_temp > 0, \
    f'некорректный style_temp={style_temp}; не запускать GRPO'
cfg = RewardConfig(style_center=style_center, style_temp=style_temp)
rm_cal = DepressionReward(cfg)
print(f'style_center (калиброванный): {style_center:.4f}, style_temp: {style_temp}')
print(f"средние по baseline: reward={np.mean([b['reward'] for b in bd]):.3f}, "
      f"слов={np.mean([b['word_count'] for b in valid]):.0f}, "
      f"antisem_rate={np.mean([b['antisem_rate'] for b in valid]):.4f}")

calibration_rows = [
    {'prompt_index': index, 'prompt': prompt, 'seed': BASELINE_SEED,
     'completion': text, **breakdown}
    for index, (prompt, text, breakdown) in
    enumerate(zip(calibration_prompts, baseline_texts, bd))
]
write_jsonl_atomic(RUN_DIR / 'baseline_essays.jsonl', calibration_rows)
calibration_summary = {
    'valid_count': len(valid), 'total_count': len(bd),
    'raw_mean': style_center, 'raw_std': float(np.std(raws)),
    'raw_min': float(np.min(raws)), 'raw_max': float(np.max(raws)),
    'style_center': style_center, 'style_temp': style_temp,
}
write_json_atomic(RUN_DIR / 'calibration.json', calibration_summary)
update_manifest(status='calibrated', calibration=calibration_summary,
                reward_config=asdict(cfg))


In [ ]:
from depression_reward.validation import markers_report

def summarize(name, texts, scorer):
    breakdown = scorer.breakdown(texts)
    valid_rows = [row for row in breakdown if not row['floored']]
    assert valid_rows, f'{name}: все тексты зафлорены'
    summary = {
        'count': len(breakdown), 'valid_count': len(valid_rows),
        'reward_mean': float(np.mean([row['reward'] for row in breakdown])),
        'raw_mean': float(np.mean([row['raw_score'] for row in valid_rows])),
        'style_mean': float(np.mean([row['style'] for row in valid_rows])),
        'antisem_rate_mean': float(np.mean([row['antisem_rate'] for row in valid_rows])),
        'word_count_mean': float(np.mean([row['word_count'] for row in valid_rows])),
        'floored_count': len(breakdown) - len(valid_rows),
    }
    print(f"{name:16s} reward={summary['reward_mean']:.3f}  "
          f"raw={summary['raw_mean']:.4f}  style={summary['style_mean']:.3f}  "
          f"antisem={summary['antisem_rate_mean']:.4f}  "
          f"слов={summary['word_count_mean']:.0f}  floored={summary['floored_count']}")
    return breakdown, summary

print('=== Маркеры calibration baseline ===')
print(markers_report([t for t, row in zip(baseline_texts, bd) if not row['floored']]))

# Важно: held-out baseline фиксируется до trainer.train().
baseline_eval, eval_pairs = generate_eval(eval_run_prompts)
bd_base, baseline_eval_summary = summarize('baseline (eval)', baseline_eval, rm_cal)
baseline_eval_rows = [
    {**pair, 'completion': text, **breakdown}
    for pair, text, breakdown in zip(eval_pairs, baseline_eval, bd_base)
]
write_jsonl_atomic(RUN_DIR / 'baseline_eval_essays.jsonl', baseline_eval_rows)
write_json_atomic(RUN_DIR / 'baseline_eval_summary.json', baseline_eval_summary)
print('=== Маркеры held-out baseline ===')
print(markers_report([text for text, row in zip(baseline_eval, bd_base)
                      if not row['floored']]))
update_manifest(status='baseline_eval_ready',
                baseline_eval=baseline_eval_summary)

### GRPO

Reward — калиброванный `depression_style_reward`; breakdown каждого шага пишется в `reward_log.jsonl` (следите, за счёт чего растёт reward: style или обход штрафов).

In [ ]:
from depression_reward import make_reward_func

REWARD_LOG_PATH = RUN_DIR / 'reward_log.jsonl'
reward_func = make_reward_func(cfg, log_path=str(REWARD_LOG_PATH))


In [ ]:
lens = [len(tokenizer.apply_chat_template([{'role': 'user', 'content': p}],
                                          add_generation_prompt=True, tokenize=True))
        for p in train_prompts]
max_prompt_length = max(lens) + 1
available_completion_length = max_seq_length - max_prompt_length
max_completion_length = (min(256, available_completion_length)
                         if SMOKE_RUN else available_completion_length)
print('max_prompt_length =', max_prompt_length,
      '| max_completion_length =', max_completion_length)

vllm_sampling_params = SamplingParams(
    min_p = 0.1, top_p = 1.0, top_k = -1, seed = TRAINING_SEED,
    stop = [tokenizer.eos_token], include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    seed = TRAINING_SEED,
    data_seed = TRAINING_SEED,
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 1e-5,          # 5e-6 за 100 шагов почти не сдвинул политику (KL ~0.001)
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations = GRPO_NUM_GENERATIONS,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = GRPO_MAX_STEPS,
    save_steps = GRPO_SAVE_STEPS,
    report_to = "none",
    output_dir = str(RUN_DIR / 'checkpoints'),
)
training_config = training_args.to_dict()
update_manifest(status='training_configured', grpo=training_config)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],
    args = training_args,
    train_dataset = train_dataset,
)
update_manifest(status='training')
try:
    train_result = trainer.train()
except BaseException as error:
    update_manifest(status='failed', failure={
        'type': type(error).__name__, 'message': str(error),
        'stage': 'trainer.train',
    })
    raise
trainer.state.save_to_json(str(RUN_DIR / 'trainer_state.json'))
update_manifest(status='trained', train_metrics=train_result.metrics)

In [ ]:
LORA_DIR = RUN_DIR / 'grpo_depression_lora'
model.save_lora(str(LORA_DIR))
update_manifest(status='lora_saved', lora_path=LORA_DIR.name)

### Оценка: baseline vs GRPO на отложенных промптах

Сравниваем заранее сохранённый baseline и GRPO на одинаковых prompt/seed-парах. Выводим описательные парные разности; выбор статистического критерия остаётся отдельным методологическим решением. Предметный критерий успеха: raw/style выросли, грамматические маркеры сдвинулись к целевому стилю, а `sentiment` и `antisem_rate` не ушли в депрессивную лексику.

In [ ]:
trained_eval, trained_pairs = generate_eval(
    eval_run_prompts, lora_request=model.load_lora(str(LORA_DIR)))
assert trained_pairs == eval_pairs, 'baseline/GRPO пары рассинхронизированы'
bd_grpo, trained_eval_summary = summarize('GRPO (eval)', trained_eval, rm_cal)

grpo_eval_rows = [
    {**pair, 'completion': text, **breakdown}
    for pair, text, breakdown in zip(eval_pairs, trained_eval, bd_grpo)
]
write_jsonl_atomic(RUN_DIR / 'grpo_eval_essays.jsonl', grpo_eval_rows)
write_json_atomic(RUN_DIR / 'grpo_eval_summary.json', trained_eval_summary)

paired_rows = []
for pair, base_text, trained_text, base_row, trained_row in zip(
        eval_pairs, baseline_eval, trained_eval, bd_base, bd_grpo):
    both_valid = not base_row['floored'] and not trained_row['floored']
    paired_rows.append({
        **pair, 'both_valid': both_valid,
        'baseline_completion': base_text, 'grpo_completion': trained_text,
        'baseline': base_row, 'grpo': trained_row,
        'raw_delta': (trained_row['raw_score'] - base_row['raw_score'])
                     if both_valid else None,
    })
write_jsonl_atomic(RUN_DIR / 'eval_pairs.jsonl', paired_rows)
valid_deltas = np.asarray([row['raw_delta'] for row in paired_rows
                           if row['raw_delta'] is not None], dtype=float)
assert len(valid_deltas), 'нет валидных baseline/GRPO пар'
paired_comparison = {
    'valid_pair_count': int(len(valid_deltas)),
    'raw_delta_mean': float(np.mean(valid_deltas)),
    'raw_delta_median': float(np.median(valid_deltas)),
    'positive_pair_count': int(np.sum(valid_deltas > 0)),
}
write_json_atomic(RUN_DIR / 'paired_comparison.json', paired_comparison)
print(f"\nПарное описательное сравнение: Δraw mean="
      f"{paired_comparison['raw_delta_mean']:+.4f}, median="
      f"{paired_comparison['raw_delta_median']:+.4f}, положительных пар "
      f"{paired_comparison['positive_pair_count']}/"
      f"{paired_comparison['valid_pair_count']}")
print('Статистическая значимость здесь намеренно не объявляется.')

print('\n=== Маркеры baseline (eval) ===')
print(markers_report([text for text, row in zip(baseline_eval, bd_base)
                      if not row['floored']]))
print('\n=== Маркеры GRPO (eval) ===')
print(markers_report([text for text, row in zip(trained_eval, bd_grpo)
                      if not row['floored']]))
update_manifest(status='eval_complete', trained_eval=trained_eval_summary,
                paired_comparison=paired_comparison)


In [ ]:
print('=== Пример эссе после GRPO ===')
print(trained_eval[0][:1500])

print('Сохранено eval-пар:', len(paired_rows), '| каталог:', RUN_DIR)


### Сохранение артефактов

Скачайте `grpo_artifacts_<run_id>.zip` из Kaggle Output. ZIP содержит manifest, тексты, eval, reward/trainer logs и LoRA, но не дублирует checkpoints. Промежуточные JSON/JSONL сохраняются сразу после каждого этапа.

In [ ]:
artifact_hashes = {}
for path in sorted(RUN_DIR.rglob('*')):
    relative = path.relative_to(RUN_DIR)
    if (path.is_file() and 'checkpoints' not in relative.parts
            and path != RUN_MANIFEST_PATH):
        artifact_hashes[relative.as_posix()] = sha256_file(path)
update_manifest(status='complete', artifact_sha256=artifact_hashes,
                completed_utc=datetime.now(timezone.utc).isoformat())

ARTIFACT_ZIP = Path(f'grpo_artifacts_{RUN_ID}.zip')
with zipfile.ZipFile(ARTIFACT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        relative = path.relative_to(RUN_DIR)
        if path.is_file() and 'checkpoints' not in relative.parts:
            archive.write(path, Path(RUN_DIR.name) / relative)
print(ARTIFACT_ZIP, ARTIFACT_ZIP.stat().st_size, 'байт')